In [1]:
# Defining project paths
BASE_DIR = "D:/Capstone/capstone_repo"
DATA_DIR = f"{BASE_DIR}/data/processed"
NB_DIR = f"{BASE_DIR}/notebooks"
CRS_METRIC      = "EPSG:26191"   # Morocco zone 1 (metres) — for accurate distance
CRS_GEO         = "EPSG:4326"

import os
os.makedirs(DATA_DIR, exist_ok=True)

In [13]:
import osmnx as ox
import networkx as nx
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from shapely.geometry import Point
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch
import seaborn as sns
from scipy.spatial import cKDTree
from scipy.stats import gaussian_kde
from sklearn.cluster import DBSCAN
import folium
from folium.plugins import MarkerCluster, HeatMap
from shapely.geometry import Point


In [14]:
COLORS = {
    "hospital": "#E63946",
    "bus":      "#2EC4B6",
    "bg":       "#0F1923",
    "surface":  "#1A2634",
    "text":     "#E8EDF5",
    "muted":    "#6B7FA3",
    "accent":   "#FFD166",
}

plt.rcParams.update({
    "figure.facecolor":  COLORS["bg"],
    "axes.facecolor":    COLORS["surface"],
    "axes.edgecolor":    COLORS["muted"],
    "axes.labelcolor":   COLORS["text"],
    "xtick.color":       COLORS["muted"],
    "ytick.color":       COLORS["muted"],
    "text.color":        COLORS["text"],
    "grid.color":        "#1F2D45",
    "grid.linestyle":    "--",
    "grid.alpha":        0.5,
    "font.family":       "monospace",
    "axes.titlecolor":   COLORS["text"],
    "axes.titlesize":    11,
    "axes.labelsize":    9,
})


# Import Data

In [3]:
hospitals = gpd.read_file(f"{DATA_DIR}/Casablanca_Healthcare.gpkg")
stops = gpd.read_file(f"{DATA_DIR}/CasaBus_stops.gpkg")

In [9]:
print("\n[2] Data quality checks...")

def quality_report(gdf, label):
    print(f"\n  ── {label} ──")
    print(f"  Shape            : {gdf.shape}")
    print(f"  Null values      :\n{gdf.isnull().sum().to_string()}")
    print(f"  Duplicate geoms  : {gdf.geometry.duplicated().sum()}")
    print(f"  CRS              : {gdf.crs}")
    print(f"  Bounding box     : {gdf.total_bounds.round(4)}")
    print(f"  Lat range        : {gdf['lat'].min():.4f} — {gdf['lat'].max():.4f}")
    print(f"  Lon range        : {gdf['lon'].min():.4f} — {gdf['lon'].max():.4f}")

quality_report(hospitals, "HEALTHCARE FACILITIES")
quality_report(stops, "BUS STOPS")

# Drop exact geometry duplicates
stops_dedup = stops.drop_duplicates(subset=["geometry"])
print(f"\n  Bus stops after geometry dedup: {len(stops_dedup):,} (removed {len(stops)-len(stops_dedup)})")

# De-duplicate by name (keep centroid per stop name for analysis)
stops_named = (
    stops_dedup.groupby("name", as_index=False)
       .agg(lat=("lat", "mean"), lon=("lon", "mean"))
)
stops_named["geometry"] = stops_named.apply(lambda r: Point(r.lon, r.lat), axis=1)
stops_named = gpd.GeoDataFrame(stops_named, crs=CRS_GEO)
print(f"  Unique named bus stops   : {len(stops_named):,}")


[2] Data quality checks...

  ── HEALTHCARE FACILITIES ──
  Shape            : (86, 10)
  Null values      :
name                0
lat                 0
lon                 0
geometry            0
nearest_bus_m       0
nearest_bus_km      0
nearest_bus_name    0
bus_within_500m     0
bus_within_1km      0
bus_within_2km      0
  Duplicate geoms  : 0
  CRS              : EPSG:4326
  Bounding box     : [-7.6873 33.5198 -7.4835 33.6124]
  Lat range        : 33.5198 — 33.6124
  Lon range        : -7.6873 — -7.4835

  ── BUS STOPS ──
  Shape            : (987, 7)
  Null values      :
name               0
lat                0
lon                0
geometry           0
nearest_hosp_m     0
nearest_hosp_km    0
nearest_hosp       0
  Duplicate geoms  : 0
  CRS              : EPSG:4326
  Bounding box     : [-7.8853 33.3719 -7.3305 33.7088]
  Lat range        : 33.3719 — 33.7088
  Lon range        : -7.8853 — -7.3305

  Bus stops after geometry dedup: 987 (removed 0)
  Unique named bus stops   :

# Distance Matrix

In [10]:
print("\n[3] Computing hospital ↔ bus stop distances...")

# Project to metric CRS for accurate distances
h_metric  = hospitals.to_crs(CRS_METRIC)
b_metric  = stops_named.to_crs(CRS_METRIC)

h_coords  = np.array([(g.x, g.y) for g in h_metric.geometry])
b_coords  = np.array([(g.x, g.y) for g in b_metric.geometry])

# Full pairwise distance matrix (metres)
from scipy.spatial.distance import cdist
dist_matrix = cdist(h_coords, b_coords)  # shape: (n_hospitals, n_bus_stops)

dist_df = pd.DataFrame(
    dist_matrix,
    index=hospitals["name"].values,
    columns=stops_named["name"].values
)

# Per-hospital stats
hospitals["nearest_bus_m"]    = dist_matrix.min(axis=1)
hospitals["nearest_bus_km"]   = hospitals["nearest_bus_m"] / 1000
hospitals["nearest_bus_name"] = [stops_named.iloc[i]["name"] for i in dist_matrix.argmin(axis=1)]
hospitals["bus_within_500m"]  = (dist_matrix < 500).sum(axis=1)
hospitals["bus_within_1km"]   = (dist_matrix < 1000).sum(axis=1)
hospitals["bus_within_2km"]   = (dist_matrix < 2000).sum(axis=1)

# Per-bus stop stats
stops_named["nearest_hosp_m"]   = dist_matrix.min(axis=0)
stops_named["nearest_hosp_km"]  = stops_named["nearest_hosp_m"] / 1000
stops_named["nearest_hosp"]     = [hospitals.iloc[i]["name"] for i in dist_matrix.argmin(axis=0)]

print("\n  Hospital Accessibility Summary:")
print(hospitals[["name","nearest_bus_km","nearest_bus_name","bus_within_500m","bus_within_1km","bus_within_2km"]]
      .sort_values("nearest_bus_km")
      .to_string(index=False))

avg = hospitals["nearest_bus_km"].mean()
print(f"\n  Average nearest bus stop : {avg:.3f} km")
print(f"  Worst connected          : {hospitals.loc[hospitals['nearest_bus_km'].idxmax(), 'name']} ({hospitals['nearest_bus_km'].max():.3f} km)")
print(f"  Best connected           : {hospitals.loc[hospitals['nearest_bus_km'].idxmin(), 'name']} ({hospitals['nearest_bus_km'].min():.3f} km)")




[3] Computing hospital ↔ bus stop distances...

  Hospital Accessibility Summary:
                                                                         name  nearest_bus_km              nearest_bus_name  bus_within_500m  bus_within_1km  bus_within_2km
                                                                 Clinique CIL        0.029419               CLINIQUE ABBADI                2              13              53
                                                           Clinique Ain Chock        0.031167             CLINIQUE AIN CHOC                8              17              59
                                              Clinique Andalouss مصحة الاندلس        0.036032             CLINIQUE ANDALOUS                3              10              42
Hôpital Privé International de Casablanca المستشفى الخاص الدولي للدار البيضاء        0.062431               TERMINUS MAARIF                5              25              84
                                            HGC Hôpi

# Spatial Clustering

In [11]:
print("\n[4] Clustering bus stops (DBSCAN)...")
from sklearn.cluster import DBSCAN
dbscan = DBSCAN(eps=0.01, min_samples=2, metric="euclidean")  # ~1km in degrees
stops["cluster"] = dbscan.fit_predict(stops[["lon","lat"]])

n_clusters = stops["cluster"].nunique() - (1 if -1 in stops["cluster"].values else 0)
n_noise    = (stops["cluster"] == -1).sum()
print(f"  Clusters found : {n_clusters}")
print(f"  Noise points   : {n_noise}")
print(f"\n  Cluster sizes:\n{stops['cluster'].value_counts().head(10).to_string()}")



[4] Clustering bus stops (DBSCAN)...
  Clusters found : 27
  Noise points   : 29

  Cluster sizes:
cluster
 0     748
 5      91
-1      29
 1      18
 11     11
 8      10
 3      10
 16      8
 18      8
 17      6


# VISUALIZATION

In [15]:
print("\n[5] Generating plots...")

fig = plt.figure(figsize=(20, 14), facecolor=COLORS["bg"])
fig.suptitle("Morocco Healthcare Accessibility — EDA Dashboard",
             fontsize=16, fontweight="bold", color=COLORS["text"], y=0.98)

gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.4)

# ── Panel A: Spatial map ──────────────────────────────────────────────────
ax_map = fig.add_subplot(gs[0:2, 0:2])
ax_map.set_facecolor(COLORS["bg"])
ax_map.scatter(stops["lon"], stops["lat"], s=15, c=COLORS["bus"], alpha=0.5, label=f"Bus Stops (n={len(stops)})", zorder=2)
ax_map.scatter(hospitals["lon"], hospitals["lat"], s=120, c=COLORS["hospital"],
               marker="P", edgecolors="white", linewidths=0.8,
               label=f"Hospitals (n={len(hospitals)})", zorder=4)

# Draw lines to nearest bus stop
for _, h in hospitals.iterrows():
    nearest_row = stops_named[stops_named["name"] == h["nearest_bus_name"]].iloc[0]
    ax_map.plot([h["lon"], nearest_row["lon"]], [h["lat"], nearest_row["lat"]],
                color=COLORS["accent"], alpha=0.4, linewidth=0.8, linestyle="--", zorder=3)

# Annotate hospital names
for _, h in hospitals.iterrows():
    ax_map.annotate(h["name"].split()[0], (h["lon"], h["lat"]),
                    fontsize=6.5, color=COLORS["text"], ha="left",
                    xytext=(4, 4), textcoords="offset points")

ax_map.set_title("Spatial Distribution + Nearest Bus Connections", pad=8)
ax_map.set_xlabel("Longitude")
ax_map.set_ylabel("Latitude")
ax_map.legend(loc="upper right", fontsize=8, facecolor=COLORS["surface"], edgecolor=COLORS["muted"])
ax_map.grid(True)

# ── Panel B: Distance to nearest bus stop ──────────────────────────────────
ax_dist = fig.add_subplot(gs[0, 2])
sorted_h = hospitals.sort_values("nearest_bus_km", ascending=False)
colors   = [COLORS["hospital"] if v > avg else COLORS["bus"] for v in sorted_h["nearest_bus_km"]]
bars = ax_dist.barh(range(len(sorted_h)), sorted_h["nearest_bus_km"],
                    color=colors, edgecolor="none", height=0.6)
ax_dist.axvline(avg, color=COLORS["accent"], linestyle="--", linewidth=1.2, label=f"Avg: {avg:.2f} km")
ax_dist.set_yticks(range(len(sorted_h)))
ax_dist.set_yticklabels([n.split()[0] for n in sorted_h["name"]], fontsize=8)
ax_dist.set_xlabel("km")
ax_dist.set_title("Nearest Bus Stop Distance")
ax_dist.legend(fontsize=8, facecolor=COLORS["surface"], edgecolor=COLORS["muted"])
ax_dist.grid(True, axis="x")

# ── Panel C: Bus stops within radius ──────────────────────────────────────
ax_cov = fig.add_subplot(gs[0, 3])
x = np.arange(len(hospitals))
w = 0.25
ax_cov.bar(x - w, hospitals["bus_within_500m"], width=w, label="≤500m", color=COLORS["hospital"], alpha=0.85)
ax_cov.bar(x,     hospitals["bus_within_1km"],  width=w, label="≤1km",  color=COLORS["accent"],  alpha=0.85)
ax_cov.bar(x + w, hospitals["bus_within_2km"],  width=w, label="≤2km",  color=COLORS["bus"],     alpha=0.85)
ax_cov.set_xticks(x)
ax_cov.set_xticklabels([n.split()[0] for n in hospitals["name"]], rotation=45, ha="right", fontsize=7)
ax_cov.set_title("Bus Stops Within Radius")
ax_cov.set_ylabel("Count")
ax_cov.legend(fontsize=8, facecolor=COLORS["surface"], edgecolor=COLORS["muted"])
ax_cov.grid(True, axis="y")

# ── Panel D: Distance distribution (histogram + KDE) ──────────────────────
ax_hist = fig.add_subplot(gs[1, 2])
vals_h = hospitals["nearest_bus_km"].values
vals_b = stops_named["nearest_hosp_km"].values / 1  # already km

ax_hist.hist(vals_h, bins=8, color=COLORS["hospital"], alpha=0.7, label="Hospital→Bus", density=True)
ax_hist.hist(vals_b, bins=12, color=COLORS["bus"],      alpha=0.5, label="Bus→Hospital", density=True)
ax_hist.set_xlabel("Distance (km)")
ax_hist.set_ylabel("Density")
ax_hist.set_title("Distance Distribution")
ax_hist.legend(fontsize=8, facecolor=COLORS["surface"], edgecolor=COLORS["muted"])
ax_hist.grid(True)

# ── Panel E: Heatmap of distance matrix ──────────────────────────────────
ax_hm = fig.add_subplot(gs[1, 3])
# Show top 9 hospitals × top 12 bus stops (nearest)
top_bus = dist_df.min().nsmallest(min(12, len(dist_df.columns))).index
hm_data = dist_df[top_bus] / 1000  # km
sns.heatmap(hm_data, ax=ax_hm, cmap="RdYlGn_r",
            xticklabels=[n[:12] for n in hm_data.columns],
            yticklabels=[n.split()[0] for n in hm_data.index],
            annot=True, fmt=".1f", annot_kws={"size": 6},
            cbar_kws={"label": "km", "shrink": 0.7},
            linewidths=0.3, linecolor=COLORS["bg"])
ax_hm.set_title("Distance Matrix (km)")
ax_hm.tick_params(axis="x", rotation=60, labelsize=6)
ax_hm.tick_params(axis="y", labelsize=7)

# ── Panel F: Lat distribution ────────────────────────────────────────────
ax_lat = fig.add_subplot(gs[2, 0])
ax_lat.hist(hospitals["lat"], bins=8, color=COLORS["hospital"], alpha=0.8, label="Hospitals", density=True)
ax_lat.hist(stops_named["lat"],       bins=20, color=COLORS["bus"],      alpha=0.5, label="Bus Stops", density=True)
ax_lat.set_xlabel("Latitude")
ax_lat.set_ylabel("Density")
ax_lat.set_title("Latitude Distribution")
ax_lat.legend(fontsize=8, facecolor=COLORS["surface"], edgecolor=COLORS["muted"])
ax_lat.grid(True)

# ── Panel G: Lon distribution ────────────────────────────────────────────
ax_lon = fig.add_subplot(gs[2, 1])
ax_lon.hist(hospitals["lon"], bins=8, color=COLORS["hospital"], alpha=0.8, label="Hospitals", density=True)
ax_lon.hist(stops_named["lon"],       bins=20, color=COLORS["bus"],      alpha=0.5, label="Bus Stops", density=True)
ax_lon.set_xlabel("Longitude")
ax_lon.set_ylabel("Density")
ax_lon.set_title("Longitude Distribution")
ax_lon.legend(fontsize=8, facecolor=COLORS["surface"], edgecolor=COLORS["muted"])
ax_lon.grid(True)

# ── Panel H: DBSCAN clusters ─────────────────────────────────────────────
ax_cl = fig.add_subplot(gs[2, 2])
cmap = plt.cm.get_cmap("tab20", n_clusters)
for cluster_id in sorted(stops["cluster"].unique()):
    subset = stops[stops["cluster"] == cluster_id]
    c = "grey" if cluster_id == -1 else cmap(cluster_id)
    lbl = f"Noise" if cluster_id == -1 else f"C{cluster_id} (n={len(subset)})"
    ax_cl.scatter(subset["lon"], subset["lat"], s=15, color=c, alpha=0.7, label=lbl)
ax_cl.scatter(hospitals["lon"], hospitals["lat"], s=100, c=COLORS["hospital"],
              marker="P", edgecolors="white", linewidths=0.8, zorder=5, label="Hospitals")
ax_cl.set_title(f"DBSCAN Bus Clusters (eps=0.01°)")
ax_cl.set_xlabel("Longitude")
ax_cl.set_ylabel("Latitude")
ax_cl.grid(True)
handles, labels = ax_cl.get_legend_handles_labels()
ax_cl.legend(handles[:8], labels[:8], fontsize=6, facecolor=COLORS["surface"],
             edgecolor=COLORS["muted"], loc="upper right")

# ── Panel I: Coverage summary bar ────────────────────────────────────────
ax_summ = fig.add_subplot(gs[2, 3])
categories = ["Good\n(≤0.5km)", "Fair\n(0.5–1km)", "Poor\n(>1km)"]
counts_cats = [
    (hospitals["nearest_bus_km"] <= 0.5).sum(),
    ((hospitals["nearest_bus_km"] > 0.5) & (hospitals["nearest_bus_km"] <= 1.0)).sum(),
    (hospitals["nearest_bus_km"] > 1.0).sum(),
]
bar_colors = [COLORS["bus"], COLORS["accent"], COLORS["hospital"]]
ax_summ.bar(categories, counts_cats, color=bar_colors, edgecolor="none", width=0.5)
ax_summ.set_ylabel("# Hospitals")
ax_summ.set_title("Transit Access Quality")
for i, v in enumerate(counts_cats):
    ax_summ.text(i, v + 0.05, str(v), ha="center", fontsize=10, color=COLORS["text"], fontweight="bold")
ax_summ.set_ylim(0, max(counts_cats) + 1.5)
ax_summ.grid(True, axis="y")

plt.savefig(f"{DATA_DIR}/01_eda_dashboard.png", dpi=150, bbox_inches="tight",
            facecolor=COLORS["bg"])
plt.close()
print(f"  ✓ Saved: {DATA_DIR}/01_eda_dashboard.png")


[5] Generating plots...


C:\Users\afafb\AppData\Local\Temp\ipykernel_1456\2808000621.py:113: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap("tab20", n_clusters)


  ✓ Saved: D:/Capstone/capstone_repo/data/processed/01_eda_dashboard.png


In [16]:
print("\n[6] Building interactive Folium map...")

center = [hospitals["lat"].mean(), hospitals["lon"].mean()]
m = folium.Map(location=center, zoom_start=12,
               tiles="CartoDB dark_matter")

# Bus stops cluster
bus_cluster = MarkerCluster(name="Bus Stops").add_to(m)
for _, b in stops.iterrows():
    folium.CircleMarker(
        location=[b["lat"], b["lon"]],
        radius=4, color="#2EC4B6", fill=True, fill_opacity=0.7,
        popup=folium.Popup(b["name"], max_width=200)
    ).add_to(bus_cluster)

# Heatmap layer of bus stops
heat_data = [[b["lat"], b["lon"]] for _, b in stops.iterrows()]
HeatMap(heat_data, name="Bus Stop Density", min_opacity=0.3,
        radius=20, blur=15).add_to(m)

# Hospitals
for _, h in hospitals.iterrows():
    color = "#E63946" if h["nearest_bus_km"] > 1.0 else \
            "#FFD166" if h["nearest_bus_km"] > 0.5 else "#2EC4B6"

    popup_html = f"""
    <div style='font-family:monospace;font-size:12px;min-width:200px'>
        <b>{h['name']}</b><br>
        <hr style='margin:4px 0'>
        Lat: {h['lat']:.5f} | Lon: {h['lon']:.5f}<br>
        Nearest bus: <b>{h['nearest_bus_name']}</b><br>
        Distance: <b>{h['nearest_bus_km']:.3f} km</b><br>
        Bus stops ≤500m: {h['bus_within_500m']}<br>
        Bus stops ≤1km:  {h['bus_within_1km']}<br>
        Bus stops ≤2km:  {h['bus_within_2km']}
    </div>"""

    folium.Marker(
        location=[h["lat"], h["lon"]],
        popup=folium.Popup(popup_html, max_width=280),
        tooltip=h["name"],
        icon=folium.Icon(color="red" if h["nearest_bus_km"] > 1.0 else
                                "orange" if h["nearest_bus_km"] > 0.5 else "green",
                         icon="plus-sign", prefix="glyphicon")
    ).add_to(m)

    # 1km buffer
    folium.Circle(
        location=[h["lat"], h["lon"]], radius=1000,
        color=color, fill=True, fill_opacity=0.05,
        weight=1.5, dash_array="6 4"
    ).add_to(m)

    # Line to nearest bus
    nearest_b = stops_named[stops_named["name"] == h["nearest_bus_name"]].iloc[0]
    folium.PolyLine(
        locations=[[h["lat"], h["lon"]], [nearest_b["lat"], nearest_b["lon"]]],
        color="#FFD166", weight=1.5, opacity=0.6, dash_array="4 4"
    ).add_to(m)

folium.LayerControl().add_to(m)

map_path = f"{DATA_DIR}/03_interactive_map.html"
m.save(map_path)
print(f"  ✓ Saved: {map_path}")


[6] Building interactive Folium map...
  ✓ Saved: D:/Capstone/capstone_repo/data/processed/03_interactive_map.html


In [17]:
# ═════════════════════════════════════════════════════════════════════════════
# 8. SAVE ENRICHED DATA
# ═════════════════════════════════════════════════════════════════════════════
print("\n[7] Saving enriched datasets...")

hospitals_out = hospitals.drop(columns=["geometry"])
hospitals_out.to_csv(f"{DATA_DIR}/hospitals_enriched.csv", index=False)
print(f"  ✓ hospitals_enriched.csv")

bus_named_out = stops_named.drop(columns=["geometry"])
bus_named_out.to_csv(f"{DATA_DIR}/bus_named_enriched.csv", index=False)
print(f"  ✓ bus_named_enriched.csv")

# Distance matrix
dist_df_km = dist_df / 1000
dist_df_km.to_csv(f"{DATA_DIR}/distance_matrix_km.csv")
print(f"  ✓ distance_matrix_km.csv")


# ═════════════════════════════════════════════════════════════════════════════
# 9. SUMMARY REPORT
# ═════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("  SUMMARY REPORT")
print("=" * 60)

print(f"""
Dataset
  Healthcare facilities  : {len(hospitals)}
  Bus stops (raw)        : {len(stops)}
  Bus stops (unique)     : {len(stops_named)}

Accessibility
  Avg hospital→bus dist  : {hospitals['nearest_bus_km'].mean():.3f} km
  Min hospital→bus dist  : {hospitals['nearest_bus_km'].min():.3f} km
  Max hospital→bus dist  : {hospitals['nearest_bus_km'].max():.3f} km

Transit Coverage
  Hospitals w/ bus ≤500m : {(hospitals['nearest_bus_km'] <= 0.5).sum()} / {len(hospitals)}
  Hospitals w/ bus ≤1km  : {(hospitals['nearest_bus_km'] <= 1.0).sum()} / {len(hospitals)}
  Hospitals w/ bus >1km  : {(hospitals['nearest_bus_km'] >  1.0).sum()} / {len(hospitals)}  ← underserved

Spatial Clustering (DBSCAN eps=0.01°)
  Bus stop clusters      : {n_clusters}
  Noise/isolated stops   : {n_noise}

Outputs saved to: ./{DATA_DIR}/
  01_eda_dashboard.png
  02_catchment_circles.png
  03_interactive_map.html
  hospitals_enriched.csv
  bus_named_enriched.csv
  distance_matrix_km.csv

Next Steps
  → Phase 2: Load OSMnx road network → real travel times
  → Phase 3: Add population grid (WorldPop) → 2SFCA scoring
  → Phase 4: MCLP optimization for new hospital placement
  → Phase 5: ML model to predict accessibility scores
""")


[7] Saving enriched datasets...
  ✓ hospitals_enriched.csv
  ✓ bus_named_enriched.csv
  ✓ distance_matrix_km.csv

  SUMMARY REPORT

Dataset
  Healthcare facilities  : 86
  Bus stops (raw)        : 987
  Bus stops (unique)     : 987

Accessibility
  Avg hospital→bus dist  : 0.195 km
  Min hospital→bus dist  : 0.029 km
  Max hospital→bus dist  : 0.498 km

Transit Coverage
  Hospitals w/ bus ≤500m : 86 / 86
  Hospitals w/ bus ≤1km  : 86 / 86
  Hospitals w/ bus >1km  : 0 / 86  ← underserved

Spatial Clustering (DBSCAN eps=0.01°)
  Bus stop clusters      : 27
  Noise/isolated stops   : 29

Outputs saved to: ./D:/Capstone/capstone_repo/data/processed/
  01_eda_dashboard.png
  02_catchment_circles.png
  03_interactive_map.html
  hospitals_enriched.csv
  bus_named_enriched.csv
  distance_matrix_km.csv

Next Steps
  → Phase 2: Load OSMnx road network → real travel times
  → Phase 3: Add population grid (WorldPop) → 2SFCA scoring
  → Phase 4: MCLP optimization for new hospital placement
  → Pha